In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 2 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    print(f"Using GPU {gpu_num}")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

import pandas as pd
from data_preprocessing import preprocess_data, preprocess_data_channel
from mlp.mlp_model import train_mlp, train_mlp_channel
from evaluation import print_metrics, print_metrics_multiclass
from visualization import plot_confusion_matrix, plot_roc_curve, plot_confusion_matrix_multiclass, plot_roc_curve_multiclass


Using GPU 2


E0000 00:00:1746734902.559192 1305751 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746734902.564948 1305751 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Load the datasets
df_tdl_cdl = pd.read_csv("dataset_tdl_cdl_training.csv")
df_tdl_cdl_test = pd.read_csv("dataset_tdl_cdl_testing.csv")

# Make the dataset for tdl only and cdl only (based on the column name model_type)
df_tdl = df_tdl_cdl[df_tdl_cdl['model_type'] == 'tdl']
df_cdl = df_tdl_cdl[df_tdl_cdl['model_type'] == 'cdl']
df_tdl_test = df_tdl_cdl_test[df_tdl_cdl_test['model_type'] == 'tdl']
df_cdl_test = df_tdl_cdl_test[df_tdl_cdl_test['model_type'] == 'cdl']

features_tdl, labels_tdl = preprocess_data_channel(df_tdl)
features_cdl, labels_cdl = preprocess_data_channel(df_cdl)
features_tdl_test, labels_tdl_test = preprocess_data_channel(df_tdl_test)
features_cdl_test, labels_cdl_test = preprocess_data_channel(df_cdl_test)

/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_abs"] = features["mean"].apply(lambda x: abs(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features["mean_phase"] = features["mean"].apply(lambda x: cmath.phase(complex(x)))
/home/l2nguyen/long_nguyen_thesis/data_preprocessing.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [ ]:

# Train the model and get predictions
y_tdl_train, y_tdl_train_pred, y_tdl_train_prob, y_tdl_test, y_tdl_test_pred, y_tdl_test_prob = train_mlp_channel(features_tdl, labels_tdl, features_tdl_test, labels_tdl_test, model_name="TDL.h5")
y_cdl_train, y_cdl_train_pred, y_cdl_train_prob, y_cdl_test, y_cdl_test_pred, y_cdl_test_prob = train_mlp_channel(features_cdl, labels_cdl, features_cdl_test, labels_cdl_test, model_name="CDL.h5")

# Define class labels
# class_labels = ['A', 'B', 'C', 'D', 'E']
# class labels must be extracted from the dataset from model_types + "_" + labels , where     labels = df["scenario"] and  model_types = df["model_type"]
class_labels = df_tdl_cdl["model_type"] + "_" + df_tdl_cdl["scenario"].astype(str).unique()


# Plot confusion matrix and ROC curve for tdl training set
plot_confusion_matrix_multiclass(y_tdl_train, y_tdl_train_pred, classes=class_labels, title='Confusion Matrix - TDL Training Set')
plot_roc_curve_multiclass(y_tdl_train, y_tdl_train_prob, classes=class_labels, title='ROC Curve - TDL Training Set')

# Print metrics for tdl training set
print("Metrics for TDL Training Set:")
print_metrics_multiclass(y_tdl_train, y_tdl_train_pred, y_tdl_train_prob)

# Plot confusion matrix and ROC curve for tdl test set
plot_confusion_matrix_multiclass(y_tdl_test, y_tdl_test_pred, classes=class_labels, title='Confusion Matrix - TDL Test Set')
plot_roc_curve_multiclass(y_tdl_test, y_tdl_test_prob, classes=class_labels, title='ROC Curve - TDL Test Set')

# Print metrics for tdl test set
print("Metrics for CDL Test Set:")
print_metrics_multiclass(y_tdl_test, y_tdl_test_pred, y_tdl_test_prob)

# Plot confusion matrix and ROC curve for cdl training set
plot_confusion_matrix_multiclass(y_cdl_train, y_cdl_train_pred, classes=class_labels, title='Confusion Matrix - CDL Training Set')
plot_roc_curve_multiclass(y_cdl_train, y_cdl_train_prob, classes=class_labels, title='ROC Curve - CDL Training Set')

# Print metrics for cdl training set
print("Metrics for CDL Training Set:")
print_metrics_multiclass(y_cdl_train, y_cdl_train_pred, y_cdl_train_prob)

# Plot confusion matrix and ROC curve for cdl test set
plot_confusion_matrix_multiclass(y_cdl_test, y_cdl_test_pred, classes=class_labels, title='Confusion Matrix - CDL Test Set')
plot_roc_curve_multiclass(y_cdl_test, y_cdl_test_prob, classes=class_labels, title='ROC Curve - CDL Test Set')

/home/l2nguyen/miniconda3/envs/mlp/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1746734907.955034 1305751 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14791 MB memory:  -> device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:86:00.0, compute capability: 7.0


Epoch 1/1000


I0000 00:00:1746734910.166343 1306022 service.cc:148] XLA service 0x7fc79c004880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746734910.166509 1306022 service.cc:156]   StreamExecutor device (0): Tesla V100-PCIE-16GB, Compute Capability 7.0
I0000 00:00:1746734910.383048 1306022 cuda_dnn.cc:529] Loaded cuDNN version 90300


 88/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2437 - loss: 1.5997

I0000 00:00:1746734911.208886 1306022 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2703 - loss: 1.5467 - val_accuracy: 0.3068 - val_loss: 1.4540 - learning_rate: 0.0010
Epoch 2/1000
306/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2959 - loss: 1.4596
Epoch 2: saving model to mlp_checkpoints/mlp_model_epoch_02.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2960 - loss: 1.4594 - val_accuracy: 0.3154 - val_loss: 1.4240 - learning_rate: 0.0010
Epoch 3/1000
297/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3125 - loss: 1.4317
Epoch 3: saving model to mlp_checkpoints/mlp_model_epoch_03.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3129 - loss: 1.4312 - val_accuracy: 0.3288 - val_loss: 1.3890 - learning_rate: 0.0010
Epoch 4/1000
297/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3214 - loss: 1.4024
Epoch 4: saving model to mlp_checkpoints/mlp_model_epoch_04.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3217 - loss: 1.4023 - val_accuracy: 0.3328 - val_loss: 1.4281 - learning_rate: 0.0010
Epoch 5/1000
310/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3411 - loss: 1.3868
Epoch 5: saving model to mlp_checkpoints/mlp_model_epoch_05.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3410 - loss: 1.3868 - val_accuracy: 0.3258 - val_loss: 1.3815 - learning_rate: 0.0010
Epoch 6/1000
242/313 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3481 - loss: 1.3745

In [ ]:
import pandas as pd
from data_preprocessing import preprocess_data
from mlp.mlp_model import train_mlp
from evaluation import print_metrics
from visualization import plot_confusion_matrix, plot_roc_curve

# Load the datasets
df_siso = pd.read_csv("dataset_sionna_siso.csv")
# df_mimo = pd.read_csv("data/dataset_sionna_mimo.csv")
# df_tdl_cdl = pd.read_csv("data/dataset_tdl_cdl.csv")

# Combine the datasets
df = df_siso
# take 80 percent of the data for training, 20 percent for testing
features, labels = preprocess_data(df.sample(frac=0.8))
features_test, labels_test = preprocess_data(df.sample(frac=0.2))
# Train the model and get predictions
y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob = train_mlp(features, labels, features_test, labels_test)

# Plot confusion matrix and ROC curve for training set
plot_confusion_matrix(y_train, y_train_pred, title='Confusion Matrix - Training Set')
plot_roc_curve(y_train, y_train_prob, title='ROC Curve - Training Set')

# Print metrics for training set
print("Metrics for Training Set:")
print_metrics(y_train, y_train_pred, y_train_prob)

# Plot confusion matrix and ROC curve for test set
plot_confusion_matrix(y_test, y_test_pred, title='Confusion Matrix - Test Set')
plot_roc_curve(y_test, y_test_prob, title='ROC Curve - Test Set')

# Print metrics for test set
print("Metrics for Test Set:")
print_metrics(y_test, y_test_pred, y_test_prob)

In [ ]:
import pandas as pd
from mlp.mlp_model import train_mlp
from evaluation import print_metrics
from visualization import plot_confusion_matrix, plot_roc_curve


df_mimo_los_training = pd.read_csv("dataset_sionna_mimo_los.csv")
df_mimo_nlos_training = pd.read_csv("dataset_sionna_mimo_nlos.csv")
df_mimo_los_testing = pd.read_csv("dataset_sionna_mimo_los_test.csv")
df_mimo_nlos_testing = pd.read_csv("dataset_sionna_mimo_nlos_test.csv")
# concatenate the datasets
df_mimo = pd.concat([df_mimo_los_training, df_mimo_nlos_training])
df_mimo_test = pd.concat([df_mimo_los_testing, df_mimo_nlos_testing])

features, labels = preprocess_data(df_mimo)
features_test, labels_test = preprocess_data(df_mimo_test)
# Preprocess the data
y_train, y_train_pred, y_train_prob, y_test, y_test_pred, y_test_prob = train_mlp(features, labels, features_test, labels_test)

# Plot confusion matrix and ROC curve for training set
plot_confusion_matrix(y_train, y_train_pred, title='Confusion Matrix - Training Set')
plot_roc_curve(y_train, y_train_prob, title='ROC Curve - Training Set')

# Print metrics for training set
print("Metrics for Training Set:")
print_metrics(y_train, y_train_pred, y_train_prob)

# Plot confusion matrix and ROC curve for test set
plot_confusion_matrix(y_test, y_test_pred, title='Confusion Matrix - Test Set')
plot_roc_curve(y_test, y_test_prob, title='ROC Curve - Test Set')

# Print metrics for test set
print("Metrics for Test Set:")
print_metrics(y_test, y_test_pred, y_test_prob)